In [1]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath('..'))

from src.features import add_datetime_features, create_lag_features, create_rolling_features

Load dữ liệu đã làm sạch từ Giai đoạn Preprocessing

In [2]:

data_path = '../data/processed/clean_data.csv'
df_clean = pd.read_csv(data_path, index_col='Datetime', parse_dates=True)

print("Kích thước dữ liệu gốc:", df_clean.shape)
display(df_clean.head(3))

Kích thước dữ liệu gốc: (9357, 13)


,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
Datetime,,,,,,,,,,,,,
2004-10-03 18:00:00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578
2004-10-03 19:00:00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255
2004-10-03 20:00:00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502


In [3]:
df_features = add_datetime_features(df_clean)

# Chọn các cột muốn tạo độ trễ và trung bình trượt
target_cols = ['CO(GT)', 'PT08.S1(CO)', 'T', 'RH'] 

# Lấy dữ liệu 3 giờ trước
df_features = create_lag_features(df_features, columns=target_cols, lags=3)

# Trung bình 6 giờ qua
df_features = create_rolling_features(df_features, columns=target_cols, window=6)
display(df_features.head(3))
print("\nDanh sách các cột sau khi trích xuất đặc trưng:")
print(df_features.columns.tolist())

Create Datetime Features ...
Đang tạo 3 Lag Features cho các cột: ['CO(GT)', 'PT08.S1(CO)', 'T', 'RH']...
Đang tạo Rolling Features (mean, window=6) cho các cột: ['CO(GT)', 'PT08.S1(CO)', 'T', 'RH']...


,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),...,T_lag_1,T_lag_2,T_lag_3,RH_lag_1,RH_lag_2,RH_lag_3,CO(GT)_rolling_mean_6,PT08.S1(CO)_rolling_mean_6,T_rolling_mean_6,RH_rolling_mean_6
Datetime,,,,,,,,,,,,,,,,,,,,,
2004-10-03 18:00:00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2004-10-03 19:00:00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,...,13.6,NaN,NaN,48.9,NaN,NaN,NaN,NaN,NaN,NaN
2004-10-03 20:00:00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,...,13.3,13.6,NaN,47.7,48.9,NaN,NaN,NaN,NaN,NaN



Danh sách các cột sau khi trích xuất đặc trưng:
['CO(GT)', 'PT08.S1(CO)', 'NMHC(GT)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'NOx(GT)', 'PT08.S3(NOx)', 'NO2(GT)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH', 'hour', 'day_of_week', 'month', 'is_weekend', 'CO(GT)_lag_1', 'CO(GT)_lag_2', 'CO(GT)_lag_3', 'PT08.S1(CO)_lag_1', 'PT08.S1(CO)_lag_2', 'PT08.S1(CO)_lag_3', 'T_lag_1', 'T_lag_2', 'T_lag_3', 'RH_lag_1', 'RH_lag_2', 'RH_lag_3', 'CO(GT)_rolling_mean_6', 'PT08.S1(CO)_rolling_mean_6', 'T_rolling_mean_6', 'RH_rolling_mean_6']


In [4]:
# Check Nan
print("Số lượng NaN sinh ra do Lag và Rolling:\n", df_features.isna().sum().tail(10))

df_final = df_features.dropna()
print(f"\nKích thước dữ liệu cuối cùng sẵn sàng train: {df_final.shape}")

os.makedirs('../data/processed', exist_ok=True)
df_final.to_csv('../data/processed/featured_data.csv')
print("Đã lưu dữ liệu có tính năng vào data/processed/featured_data.csv")

Số lượng NaN sinh ra do Lag và Rolling:
 T_lag_1                       1
T_lag_2                       2
T_lag_3                       3
RH_lag_1                      1
RH_lag_2                      2
RH_lag_3                      3
CO(GT)_rolling_mean_6         6
PT08.S1(CO)_rolling_mean_6    6
T_rolling_mean_6              6
RH_rolling_mean_6             6
dtype: int64

Kích thước dữ liệu cuối cùng sẵn sàng train: (9351, 33)
Đã lưu dữ liệu có tính năng vào data/processed/featured_data.csv
